# Wellhead Economics by Basin
## Marginal Cost Curves, Shut-in Risk, and Supply Contraction Signals

**Thesis:** At prevailing WTI, which basins are above/below breakeven? When price drops below marginal cost, wells shut in, supply contracts, and delivery gaps widen downstream.

**Data Sources:**
- **Dallas Fed Energy Survey** — Basin-level breakeven price estimates (quarterly)
- **Kansas City Fed Energy Survey** — Niobrara/DJ Basin coverage
- **EIA Drilling Productivity Report** — Production per rig, new-well rates by basin
- **EIA-914** — Monthly crude + gas production by state

**Units:**
- **$/bbl** — US dollars per barrel (breakeven cost, WTI price)
- **bbl/d** — barrels per day (production rates, production at risk)
- **Rig count** — active drilling rigs per basin (Baker Hughes methodology)

In [ ]:
# --- Colab Setup (skip if running locally) ---
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/hb-cam/commodity-flow-intelligence.git
    import os
    from getpass import getpass
    # Optional: enter your EIA API key for live data (press Enter to skip)
    _eia_key = getpass('EIA API Key (Enter to skip): ')
    if _eia_key:
        os.environ['EIA_API_KEY'] = _eia_key
        os.environ['USE_LIVE_API'] = 'true'
    _ais_key = getpass('AISstream API Key (Enter to skip): ')
    if _ais_key:
        os.environ['AISSTREAM_API_KEY'] = _ais_key
    print('Setup complete.' + (' Live mode enabled.' if _eia_key else ' Using offline data.'))

#### Setup

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from commodity_flow import config, eia, offline, analysis
from commodity_flow.provenance import ProvenanceTracker, DataSource

plt.style.use(config.PLOT_STYLE)
plt.rcParams['figure.figsize'] = config.PLOT_FIGSIZE
plt.rcParams['font.size'] = config.PLOT_FONTSIZE

prov = ProvenanceTracker()

print(f"Basins tracked: {list(config.BASINS.keys())}")
print(f"Live API mode: {config.USE_LIVE_API}")

---
## 1. Load Data

In [ ]:
# Breakeven data (Dallas/KC Fed surveys — offline, no API)
df_breakevens = offline.generate_offline_breakevens()
prov.record(DataSource("Basin breakevens", "Offline (Dallas/KC Fed)", "offline generator", False,
    rows=len(df_breakevens), notes="Quarterly survey data; no public API. Baselined to Q4 2024 values."))

# Drilling Productivity Report — not in EIA API v2, always offline
df_dpr = offline.generate_offline_dpr()
prov.record(DataSource("Drilling productivity", "Offline (EIA DPR)", "offline generator", False,
    rows=len(df_dpr), notes="DPR not in EIA API; baselined to Feb 2025 spreadsheet values"))

print(f"\u2705 Data loaded:")
print(f"  Breakevens: {len(df_breakevens)} quarterly observations across {df_breakevens['basin'].nunique()} basins")
print(f"  Drilling productivity: {len(df_dpr)} monthly observations")
print(f"\nBasins in breakeven data: {sorted(df_breakevens['basin'].unique())}")
print(f"Date range: {df_breakevens['date'].min().strftime('%Y-%m')} to {df_breakevens['date'].max().strftime('%Y-%m')}")

# Get current WTI from live market; fall back to trailing average
try:
    from commodity_flow import futures as _fut_mod
    _fut_df = _fut_mod.fetch_futures_curves()
    current_wti = round(float(_fut_df[_fut_df['symbol'] == 'CL=F'].sort_values('date').iloc[-1]['close']), 2)
    _wti_src = 'Yahoo Finance (live)'
except Exception:
    current_wti = round(float(df_breakevens['wti_price_usd_bbl'].tail(4).mean()), 2)
    _wti_src = 'breakevens trailing 4Q avg (offline)'
print(f"\nWTI reference: ${current_wti:.2f}/bbl ({_wti_src})")

# Provenance table
from IPython.display import Markdown
display(Markdown(prov.summary()))

---
## Executive Summary

In [ ]:
# Compute headline wellhead metrics
_status = analysis.compute_breakeven_status(df_breakevens, current_wti)
_n_at_risk = (_status['status'] == 'at risk').sum()
_n_marginal = (_status['status'] == 'marginal').sum()
_at_risk_names = ', '.join(_status[_status['status'] == 'at risk']['basin'].tolist()) or 'None'

# Production at risk
_latest_prod = df_dpr.sort_values('date').groupby('basin').last().reset_index()
_merged = _status.merge(_latest_prod[['basin', 'total_new_well_production_bbl_d']], on='basin', how='inner')
_total_prod = _merged['total_new_well_production_bbl_d'].sum()
_risk_prod = _merged[_merged['status'] == 'at risk']['total_new_well_production_bbl_d'].sum()
_risk_pct = _risk_prod / _total_prod * 100 if _total_prod > 0 else 0

# Rig trend
_rigs_latest = df_dpr.sort_values('date').groupby('basin')['rig_count'].last().sum()
_rigs_peak = df_dpr.groupby('basin')['rig_count'].max().sum()
_rig_decline = (_rigs_latest - _rigs_peak) / _rigs_peak * 100

# 50% at risk threshold
_rc = analysis.production_at_risk_curve(df_breakevens, df_dpr)
_fifty = _rc[_rc['pct_at_risk'] >= 50]
_wti_50 = f'${_fifty.iloc[0]["wti_price"]:.0f}' if not _fifty.empty else 'N/A'

from IPython.display import Markdown
display(Markdown(f'''
At current WTI **${current_wti:.0f}/bbl**:

| Metric | Value | Signal |
|--------|-------|--------|
| Basins below breakeven | **{_n_at_risk} of 7** ({_at_risk_names}) | {'🔴 Shut-in risk' if _n_at_risk >= 3 else '⚠️ Elevated' if _n_at_risk >= 1 else '✅ All profitable'} |
| Production at risk | **{_risk_pct:.0f}%** ({_risk_prod:,.0f} bbl/d) | {'🔴 Significant' if _risk_pct > 25 else '⚠️ Watch' if _risk_pct > 10 else '✅ Minimal'} |
| Rig count vs peak | **{_rig_decline:+.0f}%** ({_rigs_latest:.0f} active) | {'⚠️ Declining' if _rig_decline < -10 else '✅ Stable'} |
| 50% shut-in threshold | **WTI {_wti_50}** | {'\u26a0\ufe0f Close to current price' if isinstance(_wti_50, str) and _wti_50 != 'N/A' and float(_wti_50.replace('$','')) > current_wti - 15 else ''} |

> **Bottom line:** {'Basins are shutting in. Supply contraction underway \u2014 delivery gaps will widen.' if _n_at_risk >= 3 else 'Most basins profitable at current prices, but rig counts are declining. Watch for price drops below ${0:.0f} to trigger broader shut-ins.'.format(float(_wti_50.replace('$','')) if _wti_50 != 'N/A' else 50)}
'''))

---
## 2. Basin Breakeven Chart

Horizontal bar chart: breakeven price per basin with current WTI overlay.  
**Green** = profitable, **Yellow** = marginal (within $10), **Red** = below breakeven.

> **Data currency:** Breakeven estimates are from the Dallas/KC Fed Q4 2024 surveys. WTI reference price is fetched live from Yahoo Finance at runtime; the source is printed above.

In [ ]:
status = analysis.compute_breakeven_status(df_breakevens, current_wti)

fig, ax = plt.subplots(figsize=(12, 7))

colors = status["status"].map({
    "profitable": "#22c55e",
    "marginal": "#f59e0b",
    "at risk": "#ef4444"
})

bars = ax.barh(status["basin"], status["breakeven_usd_bbl"], color=colors, height=0.6, alpha=0.85)

# WTI reference line
ax.axvline(current_wti, color="#1e293b", lw=2.5, ls="--", label=f"WTI ${current_wti}/bbl")

# Margin annotations
for i, (_, row) in enumerate(status.iterrows()):
    margin = row["margin_usd_bbl"]
    sign = "+" if margin > 0 else ""
    ax.text(row["breakeven_usd_bbl"] + 1, i, f"${sign}{margin:.0f}",
            va="center", fontweight="bold", fontsize=10,
            color="#22c55e" if margin > 0 else "#ef4444")

ax.set_xlabel("Breakeven Price ($/bbl)", fontsize=12)
ax.set_title(f"Basin Breakeven vs. WTI ${current_wti}/bbl \u2014 Profitability Status",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc="lower right")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
plt.tight_layout()
plt.show()

display(status[["basin", "play", "breakeven_usd_bbl", "margin_usd_bbl", "status"]].reset_index(drop=True))

---
## 3. Production-at-Risk Analysis

At current WTI, how much US production is from basins below breakeven?

In [ ]:
# Merge breakeven status with production data
latest_prod = df_dpr.sort_values("date").groupby("basin").last().reset_index()
merged = status.merge(
    latest_prod[["basin", "total_new_well_production_bbl_d", "rig_count"]],
    on="basin", how="inner"
)

total_prod = merged["total_new_well_production_bbl_d"].sum()
at_risk = merged[~merged["profitable"]]
at_risk_prod = at_risk["total_new_well_production_bbl_d"].sum()

print(f"Total tracked production: {total_prod:,.0f} bbl/d")
print(f"Production at risk (below breakeven): {at_risk_prod:,.0f} bbl/d ({at_risk_prod/total_prod*100:.1f}%)")
print(f"Basins at risk: {', '.join(at_risk['basin'].tolist()) if not at_risk.empty else 'None'}")

# Treemap-style bar chart
fig, ax = plt.subplots(figsize=(14, 6))
sort_merged = merged.sort_values("total_new_well_production_bbl_d", ascending=True)
bar_colors = sort_merged["status"].map({
    "profitable": "#22c55e", "marginal": "#f59e0b", "at risk": "#ef4444"
})

ax.barh(sort_merged["basin"], sort_merged["total_new_well_production_bbl_d"],
        color=bar_colors, height=0.6, alpha=0.85)

for i, (_, row) in enumerate(sort_merged.iterrows()):
    pct = row["total_new_well_production_bbl_d"] / total_prod * 100
    ax.text(row["total_new_well_production_bbl_d"] + total_prod * 0.01, i,
            f"{row['total_new_well_production_bbl_d']:,.0f} bbl/d ({pct:.1f}%)",
            va="center", fontsize=9)

ax.set_xlabel("New-Well Production (bbl/d)", fontsize=12)
ax.set_title(f"Basin Production \u2014 Color = Profitability at ${current_wti}/bbl WTI",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Supply Elasticity Curve

Sweep WTI from $30 to $100 — at each price, compute cumulative production at risk.  
This is the **marginal cost curve** for US shale: the supply-side elasticity.

In [ ]:
risk_curve = analysis.production_at_risk_curve(df_breakevens, df_dpr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Absolute production at risk
ax1.fill_between(risk_curve["wti_price"], risk_curve["production_at_risk_bbl_d"],
                 alpha=0.3, color="#ef4444")
ax1.plot(risk_curve["wti_price"], risk_curve["production_at_risk_bbl_d"],
         color="#ef4444", lw=2.5)
ax1.axvline(current_wti, color="#1e293b", ls="--", lw=2, label=f"Current WTI ${current_wti}")
ax1.set_xlabel("WTI Price ($/bbl)")
ax1.set_ylabel("Production at Risk (bbl/d)")
ax1.set_title("Absolute Production at Risk", fontweight="bold")
ax1.legend()
ax1.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# Percentage at risk
ax2.fill_between(risk_curve["wti_price"], risk_curve["pct_at_risk"],
                 alpha=0.3, color="#ef4444")
ax2.plot(risk_curve["wti_price"], risk_curve["pct_at_risk"],
         color="#ef4444", lw=2.5)
ax2.axvline(current_wti, color="#1e293b", ls="--", lw=2, label=f"Current WTI ${current_wti}")
ax2.axhline(25, color="#f97316", ls=":", alpha=0.7, label="25% threshold")
ax2.set_xlabel("WTI Price ($/bbl)")
ax2.set_ylabel("% of US Production at Risk")
ax2.set_title("Supply Elasticity Curve", fontweight="bold")
ax2.legend()
ax2.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

fig.suptitle("US Shale Marginal Cost Curve \u2014 Production at Risk by WTI Price",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Key price levels
print("\n=== Key Price Levels ===")
for pct_threshold in [10, 25, 50, 75]:
    row = risk_curve[risk_curve["pct_at_risk"] >= pct_threshold]
    if not row.empty:
        price = row.iloc[0]["wti_price"]
        print(f"  {pct_threshold}% at risk below: ${price}/bbl")

---
## 5. Rig Count Trends by Basin

Trailing rig counts by basin from the Drilling Productivity Report.  
Rig counts respond to price with a lag — declining rigs = future supply contraction.

In [ ]:
basins_sorted = df_dpr.groupby("basin")["rig_count"].last().sort_values(ascending=False).index

fig, axes = plt.subplots(2, 4, figsize=(20, 9), sharey=False)
axes = axes.flatten()

basin_colors = {
    "Permian": "#2563eb", "Eagle Ford": "#16a34a", "Bakken": "#dc2626",
    "DJ/Niobrara": "#9333ea", "Appalachian": "#0891b2",
    "Haynesville": "#ca8a04", "Anadarko": "#e11d48"
}

for i, basin in enumerate(basins_sorted):
    ax = axes[i]
    sub = df_dpr[df_dpr["basin"] == basin].sort_values("date")
    color = basin_colors.get(basin, "#64748b")

    ax.plot(sub["date"], sub["rig_count"], color=color, lw=2)
    ax.fill_between(sub["date"], sub["rig_count"], alpha=0.15, color=color)

    # Mark decline periods
    sub["rig_delta_3m"] = sub["rig_count"].diff(3)
    declining = sub[sub["rig_delta_3m"] < -sub["rig_count"].mean() * 0.1]
    for _, row in declining.iterrows():
        ax.axvspan(row["date"] - pd.Timedelta(days=15),
                   row["date"] + pd.Timedelta(days=15),
                   alpha=0.15, color="#ef4444")

    ax.set_title(f"{basin}", fontweight="bold")
    ax.set_ylabel("Active Rigs")
    ax.tick_params(axis='x', rotation=30)

# Hide unused subplot
axes[7].set_visible(False)

fig.suptitle("Rig Count Trends by Basin \u2014 Red Zones = Significant Decline",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Production Efficiency — Output per Rig

Technology improvements push production per rig higher over time,  
partially offsetting rig count declines.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9), sharey=False)
axes = axes.flatten()

for i, basin in enumerate(basins_sorted):
    ax = axes[i]
    sub = df_dpr[df_dpr["basin"] == basin].sort_values("date")
    color = basin_colors.get(basin, "#64748b")

    ax.plot(sub["date"], sub["production_per_rig_bbl_d"], color=color, lw=2)

    # Trend line
    x_num = (sub["date"] - sub["date"].min()).dt.days.values
    if len(x_num) > 2:
        z = np.polyfit(x_num, sub["production_per_rig_bbl_d"].values, 1)
        p = np.poly1d(z)
        ax.plot(sub["date"], p(x_num), '--', color="#64748b", alpha=0.7, lw=1)

    ax.set_title(f"{basin}", fontweight="bold")
    ax.set_ylabel("bbl/d per rig")
    ax.tick_params(axis='x', rotation=30)

axes[7].set_visible(False)

fig.suptitle("New-Well Production per Rig \u2014 Technology vs. Geology",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Breakeven Trend Over Time

How have basin breakevens evolved? Technology lowers costs, inflation raises them.

> **Note:** Breakeven trends are limited to the offline data range (2022–2026). Earlier data would require historical Fed survey archives not available via API.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for basin in basins_sorted:
    sub = df_breakevens[df_breakevens["basin"] == basin].sort_values("date")
    color = basin_colors.get(basin, "#64748b")
    ax.plot(sub["date"], sub["breakeven_usd_bbl"], 'o-', ms=4, color=color,
            label=basin, lw=1.5)

# WTI price overlay
wti_by_quarter = df_breakevens.drop_duplicates("date").sort_values("date")
ax.plot(wti_by_quarter["date"], wti_by_quarter["wti_price_usd_bbl"],
        'k--', lw=2.5, label="WTI Price", alpha=0.7)

ax.set_ylabel("$/bbl", fontsize=12)
ax.set_title("Basin Breakeven Trends vs. WTI Price", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=9, ncol=2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
plt.tight_layout()
plt.show()

---
## 8. Double Signal: Delivery Gap + Shut-in Risk

The most actionable signal in this analysis: when **both** delivery gaps are widening (NB01) **and** basins are shutting in (above), supply contraction is accelerating faster than the market expects.

> **How to read this chart:** The red line is the composite gap score from delivery gap analysis. The purple bars show how many basins are below breakeven each quarter. When both are elevated simultaneously, the supply squeeze is compounding.

In [ ]:
# Build scorecard from import/natgas data
df_imports = offline.generate_offline_imports()
df_natgas = offline.generate_offline_natgas_imports()
df_steo = offline.generate_offline_steo()
scorecard = analysis.build_scorecard(df_imports, df_natgas, df_steo)
actual_sc = scorecard[~scorecard["is_forecast"]]

# Count basins at risk over time (from breakeven data)
risk_over_time = []
for _, qrow in df_breakevens.drop_duplicates("date").sort_values("date").iterrows():
    q_date = qrow["date"]
    wti = qrow["wti_price_usd_bbl"]
    q_data = df_breakevens[df_breakevens["date"] == q_date]
    n_at_risk = (q_data["breakeven_usd_bbl"] > wti).sum()
    risk_over_time.append({"date": q_date, "basins_at_risk": n_at_risk, "wti": wti})
df_risk = pd.DataFrame(risk_over_time).set_index("date")

# Dual-axis plot
fig, ax1 = plt.subplots(figsize=(16, 7))
ax2 = ax1.twinx()

# Gap scorecard
ax1.plot(actual_sc.index, actual_sc["composite_gap_score"],
         color="#dc2626", lw=2.5, label="Composite Gap Score")
ax1.axhline(-1, color="#f97316", ls="--", lw=0.8)
ax1.axhline(-2, color="#ef4444", ls="--", lw=0.8)
ax1.axhline(0, color="black", lw=0.3)
ax1.fill_between(actual_sc.index, -2, actual_sc["composite_gap_score"],
                 where=actual_sc["composite_gap_score"] < -2,
                 alpha=0.2, color="#ef4444", interpolate=True)
ax1.set_ylabel("Gap Score (Z)", color="#dc2626", fontsize=12)

# Basins at risk (bar chart on right axis)
ax2.bar(df_risk.index, df_risk["basins_at_risk"],
        width=60, alpha=0.4, color="#7c3aed", label="Basins Below Breakeven")
ax2.set_ylabel("Basins at Risk", color="#7c3aed", fontsize=12)
ax2.set_ylim(0, 8)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left", fontsize=10)

ax1.set_title("Delivery Gap Score + Basin Shut-in Risk \u2014 Double Signal",
              fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nWhen BOTH signals fire (gap score < -1 AND basins shutting in),")
print("supply contraction is accelerating. Watch downstream derivative prices.")

In [ ]:
from IPython.display import Markdown as _Md2

_sc2 = analysis.build_scorecard(
    offline.generate_offline_imports(),
    offline.generate_offline_natgas_imports(),
    offline.generate_offline_steo()
)
_actual2 = _sc2[~_sc2['is_forecast']]
_z2 = _actual2['composite_gap_score'].iloc[-1] if not _actual2.empty else 0

_gap_firing = _z2 < -1
_shutin_firing = _n_at_risk >= 2

_gap_icon = "\U0001f534 FIRING" if _gap_firing else "\u2705 not firing"
_shutin_icon = "\U0001f534 FIRING" if _shutin_firing else "\u2705 not firing"

_lines = [
    "### Double Signal Assessment\n",
    f"| Signal | Current | Status |",
    f"|--------|---------|--------|",
    f"| Composite gap score | {_z2:.1f}\u03c3 | {_gap_icon} |",
    f"| Basins below breakeven | {_n_at_risk}/7 | {_shutin_icon} |",
    "",
]

if _gap_firing and _shutin_firing:
    _lines += [
        "> **\U0001f534 DOUBLE SIGNAL ACTIVE** — Physical supply contraction is accelerating.\n",
        "Consider:",
        "- Long WTI/Brent calendar spreads (backwardation steepening)",
        "- Long helium equities (DME, RLPI, PHX) as co-production supply proxy",
        "- Short RBOB crack spreads if refinery crude inputs tighten",
        "- Long heating oil (HO=F) if distillate DoS < 30 days",
    ]
elif _gap_firing or _shutin_firing:
    _lines.append("> **\u26a0\ufe0f SINGLE SIGNAL** — one leg active, watch for confirmation.\n")
    if _gap_firing:
        _lines.append("Gap score firing but basins still profitable. Monitor WTI for price decline that triggers shut-ins.")
    else:
        _lines.append("Basins under pressure but imports holding. Monitor PADD-level flows for cracks.")
else:
    _lines.append("> **\u2705 Neither signal firing.** Supply conditions normal.\n")
    _lines.append("Monitor the [Market Intelligence Briefing](03_market_intelligence_briefing.ipynb) for threshold breaches.")

display(_Md2("\n".join(_lines)))
